# 02 – Laya zero-shot baseline on pilot (Kaggle)

> **Provenance note:** reconstructed from the exact Kaggle cell code used in the experiment (the original notebook was not saved to disk). Pilot only (60 rows → 144 instances). Laya is a zero-shot baseline; result: near chance, over-confident. Output → `results/gate_pilot/results_laya_pilot.csv`.

Run on Kaggle with GPU + Internet on; add the repo `data/` files as a Kaggle dataset.

In [ ]:
import os
os.environ["USE_TF"] = "0"
!pip install -q laya

In [ ]:
import glob, json, os, time
import pandas as pd

xlsx = glob.glob("/kaggle/input/**/AISIS_DevSet_Pilot_v1.xlsx", recursive=True)[0]
qjson = glob.glob("/kaggle/input/**/gate_questions_v1.json", recursive=True)[0]
print(xlsx)

dev = pd.read_excel(xlsx, sheet_name="DevSet").fillna("")
QUESTIONS = json.load(open(qjson, encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}

instances = []
for r in dev.to_dict("records"):
    base = dict(dev_id=r["dev_id"], type=r["type"], topic=r["topic"],
                gold_action=r["label_action"], gold_supported=r["label_supported"],
                gold_specific=r["label_specific"], gold_emergency=r["label_emergency"],
                gold_tool=r["label_tool"])
    instances.append({**base, "version": "standard", "query": r["query_standard"], "passage": r["passage_text"]})
    instances.append({**base, "version": "dialect", "query": r["query_dialect_synth"], "passage": r["passage_text"]})
    if r["type"] == "answerable":
        instances.append({**base, "version": "wrong_passage", "query": r["query_standard"],
                          "passage": r["wrong_passage_text"],
                          "gold_action": "cannot_answer", "gold_supported": "not_supported"})
print(len(instances), "instances")

In [ ]:
import laya
agent = laya.load("convaiinnovations/laya", subfolder="multilingual")
try:
    agent.cfg["head_max_len"] = 512
    agent.cfg["max_len"] = 2048
except Exception as e:
    print("cfg not settable:", e)

def predict(state, questions):
    res = agent.predict(state, questions)
    return res["answers"], res

In [ ]:
def state_of(inst):
    return {"user_query": inst["query"], "retrieved_passage": inst["passage"], "available_tools": TOOLS}

rows, t0 = [], time.time()
for k, inst in enumerate(instances):
    answers, raw = predict(state_of(inst), QUESTIONS)
    if k == 0:
        print(raw)
    row = {x: inst[x] for x in ["dev_id", "type", "topic", "version", "gold_action", "gold_supported",
                                "gold_specific", "gold_emergency", "gold_tool"]}
    for q in QUESTIONS:
        a = answers[q]
        row[f"pred_{q}"] = a.get("choice")
        row[f"conf_{q}"] = a.get("confidence")
        row[f"probs_{q}"] = json.dumps(a.get("probabilities"), ensure_ascii=False, default=str)
    rows.append(row)
elapsed = time.time() - t0
print(f"{elapsed:.1f}s total, {1000*elapsed/len(instances):.0f} ms per instance (5 questions each)")
try:
    import torch
    print("peak GPU memory GB:", torch.cuda.max_memory_allocated() / 1e9)
except Exception:
    pass
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/results_laya_pilot.csv", index=False)

In [ ]:
res["ok_action"] = res.pred_action == res.gold_action
res["ok_supported"] = res.pred_supported == res.gold_supported
res["ok_specific"] = res.pred_specific == res.gold_specific
res["ok_emergency"] = res.pred_emergency == res.gold_emergency
tool_rows = res[res.gold_action == "call_tool"].copy()
tool_rows["ok_tool"] = tool_rows.pred_tool == tool_rows.gold_tool
print(res.groupby("version")[["ok_action", "ok_supported", "ok_specific", "ok_emergency"]].mean().round(3))
print("\nTool accuracy (call_tool rows):\n", tool_rows.groupby("version").ok_tool.mean().round(3))
print("\nAction accuracy by type:\n", res[res.version != "wrong_passage"].groupby(["type", "version"]).ok_action.mean().round(3).unstack())